In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"                  # avoid DataParallel in notebook
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PER_DEVICE_TRAIN_BATCH_SIZE"] = "1"
os.environ["GRADIENT_ACCUMULATION_STEPS"] = "1"
os.environ["NUM_GENERATIONS"] = "2"          # much lighter than 4/6
# os.environ["MAX_PROMPT_LENGTH"] = "256"
os.environ["MAX_COMPLETION_LENGTH"] = "24"
os.environ["DATASET_SIZE"] = "256"

In [3]:
! git clone https://github.com/Kraken-coder/open_env_meta_torch_comp.git

Cloning into 'open_env_meta_torch_comp'...
remote: Enumerating objects: 437, done.
remote: Counting objects: 100% (437/437), done.
remote: Compressing objects: 100% (331/331), done.
remote: Total 437 (delta 113), reused 428 (delta 104), pack-reused 0 (from 0)
Receiving objects: 100% (437/437), 2.93 MiB | 10.90 MiB/s, done.
Resolving deltas: 100% (113/113), done.
Filtering content: 100% (232/232), 161.86 MiB | 34.73 MiB/s, done.


In [4]:
cd open_env_meta_torch_comp

/kaggle/working/open_env_meta_torch_comp


In [5]:
ls memory_action_grpo_qwen15b

adapter_config.json        checkpoint-512/  tokenizer_config.json
adapter_model.safetensors  completions/     tokenizer.json
chat_template.jinja        README.md        training_args.bin
checkpoint-1024/           ref/


In [6]:
! pip install openenv-core[core,web]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 12.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.4 MB/s eta 0:00:00


In [7]:
!pip install transformers datasets peft trl accelerate bitsandbytes openenv-core[core,web]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 11.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.3 MB/s eta 0:00:00:00:0100:01


In [8]:
! ls

benchmark_models.py	 __init__.py		     random_baseline.py
benchmark_results	 kaggle_benchmarking	     README.md
client.py		 memory_action_grpo_qwen15b  server
data.py			 memory_action_sft_qwen15b   train_grpo_memory.py
deployment_commands.txt  models.py		     train_sft_qwen.py
Dockerfile		 openenv.yaml		     uv.lock
grpo_reward_log.jsonl	 __pycache__		     validation.txt
inference.py		 pyproject.toml		     verify_sft_model.py


In [9]:
! pip install -U bitsandbytes>=0.46.1

In [11]:
"""
SFT trainer for Long Horizon Memory action generation.

Trains a LoRA adapter on top of Qwen/Qwen2.5-1.5B-Instruct using seed data in
data.py. The model learns to emit action JSON parseable by LongHorizonMemoryAction.

Usage:
    python train_sft_qwen.py

Recommended install (GPU):
    pip install -U transformers datasets peft trl accelerate bitsandbytes
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

from data import SEED_DATA, SYSTEM_PROMPT, action_to_json, format_observation


@dataclass
class TrainConfig:
    model_name: str = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
    output_dir: str = os.getenv("OUTPUT_DIR", "./memory_action_sft_qwen15b")
    max_length: int = int(os.getenv("MAX_LENGTH", "768"))
    learning_rate: float = float(os.getenv("LEARNING_RATE", "2e-4"))
    num_train_epochs: int = int(os.getenv("NUM_TRAIN_EPOCHS", "1"))
    per_device_train_batch_size: int = int(os.getenv("PER_DEVICE_TRAIN_BATCH_SIZE", "2"))
    gradient_accumulation_steps: int = int(os.getenv("GRADIENT_ACCUMULATION_STEPS", "4"))
    seed_repeat_factor: int = int(os.getenv("SEED_REPEAT_FACTOR", "5"))
    lora_r: int = int(os.getenv("LORA_R", "16"))
    lora_alpha: int = int(os.getenv("LORA_ALPHA", "32"))
    lora_dropout: float = float(os.getenv("LORA_DROPOUT", "0.05"))


CFG = TrainConfig()


def build_tokenizer() -> AutoTokenizer:
    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def build_model() -> AutoModelForCausalLM:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU is required for Qwen2.5-1.5B SFT in this script.")

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        CFG.model_name,
        trust_remote_code=True,
        quantization_config=bnb,
        device_map="auto",
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()

    peft_config = LoraConfig(
        r=CFG.lora_r,
        lora_alpha=CFG.lora_alpha,
        lora_dropout=CFG.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, peft_config)
    return model


def apply_chat_template(tokenizer: AutoTokenizer, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return (
            f"<|system|>\n{SYSTEM_PROMPT}\n"
            f"<|user|>\n{user_prompt}\n"
            "<|assistant|>\n"
        )


def build_rows(tokenizer: AutoTokenizer) -> List[Dict[str, str]]:
    rows: List[Dict[str, str]] = []
    for sample in SEED_DATA:
        obs = sample["observation"]
        action = sample["response"]
        prompt_body = format_observation(obs)
        prompt = apply_chat_template(tokenizer, prompt_body)
        target = action_to_json(action)
        rows.append({"prompt": prompt, "full_text": prompt + target})
    return rows


def tokenize_and_mask(example: Dict[str, str], tokenizer: AutoTokenizer) -> Dict[str, Any]:
    full = tokenizer(
        example["full_text"],
        truncation=True,
        max_length=CFG.max_length,
        padding="max_length",
    )
    prompt = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=CFG.max_length,
        padding="max_length",
    )

    labels = list(full["input_ids"])
    prompt_len = sum(1 for t in prompt["attention_mask"] if t == 1)
    for i in range(prompt_len):
        labels[i] = -100
    labels = [
        -100 if tok == tokenizer.pad_token_id else lab
        for tok, lab in zip(full["input_ids"], labels)
    ]
    full["labels"] = labels
    return full


def main() -> None:
    print(f"[SFT] Loading tokenizer: {CFG.model_name}")
    tokenizer = build_tokenizer()

    rows = build_rows(tokenizer)
    rows = rows * max(1, CFG.seed_repeat_factor)
    dataset = Dataset.from_list(rows)
    dataset = dataset.map(
        lambda ex: tokenize_and_mask(ex, tokenizer),
        remove_columns=dataset.column_names,
        num_proc=1,  # keep single process for broad compatibility
    )

    print(f"[SFT] Training samples: {len(dataset)}")
    print("[SFT] Loading 4-bit base model + LoRA adapters...")
    model = build_model()

    args = TrainingArguments(
        output_dir=CFG.output_dir,
        num_train_epochs=CFG.num_train_epochs,
        per_device_train_batch_size=CFG.per_device_train_batch_size,
        gradient_accumulation_steps=CFG.gradient_accumulation_steps,
        learning_rate=CFG.learning_rate,
        weight_decay=0.0,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
        fp16=False,
        bf16=False,
        max_grad_norm=1.0,
        seed=42,
        dataloader_num_workers=0,
        optim="paged_adamw_8bit",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )

    print("[SFT] Starting training...")
    trainer.train()
    trainer.save_model(CFG.output_dir)
    tokenizer.save_pretrained(CFG.output_dir)
    print(f"[SFT] Saved adapter and tokenizer to: {CFG.output_dir}")


if __name__ == "__main__":
    main()


[SFT] Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


Map (num_proc=1):   0%|          | 0/175 [00:00<?, ? examples/s]

[SFT] Training samples: 175
[SFT] Loading 4-bit base model + LoRA adapters...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[SFT] Starting training...


Step,Training Loss
10,3.400549
20,1.920982
30,1.087881
40,0.999781
50,0.706775
60,0.820002
70,0.684819
80,0.439621
90,0.267266
100,0.308504


[SFT] Saved adapter and tokenizer to: ./memory_action_sft_qwen15b


In [12]:
# Jupyter-safe GRPO trainer for Long Horizon Memory
# Includes:
# - notebook-safe path resolution
# - robust TRL config compatibility
# - generation_batch_size fix
# - reward/completion JSONL logging + console summaries
# - safer single-GPU loading for notebook stability

from __future__ import annotations

import json
import os
import random
import re
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# Long-horizon training: lift the env's memory capacity from 8 -> 16 so we
# actually exercise eviction. The env reads this env var at class-definition
# time, so it must be set before the env module is imported below.
os.environ.setdefault("LONG_HORIZON_MEMORY_CAPACITY", "16")

import torch
from datasets import Dataset
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    LogitsProcessor,
    LogitsProcessorList,
)


def _resolve_here() -> Path:
    if "__file__" in globals():
        p = Path(__file__).resolve().parent
        if (p / "server").exists() and (p / "data.py").exists():
            return p

    cwd = Path.cwd().resolve()
    if (cwd / "server").exists() and (cwd / "data.py").exists():
        return cwd
    if (cwd / "Long_Horizon_Memory" / "server").exists():
        return cwd / "Long_Horizon_Memory"

    raise RuntimeError(
        "Could not locate Long_Horizon_Memory root. "
        "Run this notebook from repo root or Long_Horizon_Memory directory."
    )


HERE = _resolve_here()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE / "server"))

from data import SYSTEM_PROMPT, format_observation  # noqa: E402
from models import LongHorizonMemoryAction  # noqa: E402
from server.long_horizon_memory_environment import (  # noqa: E402
    LongHorizonMemoryEnvironment,
    score_action,
)


@dataclass
class TrainConfig:
    model_name: str = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
    adapter_path: str = os.getenv("ADAPTER_PATH", str(HERE / "memory_action_sft_qwen15b"))
    output_dir: str = os.getenv("OUTPUT_DIR", str(HERE / "memory_action_grpo_qwen15b"))
    # Long-horizon dataset: episodes 60-110 msgs, 19-20 relevant per ep,
    # capacity is set to 16 below so memory actually fills and we get real
    # eviction pressure (remove must be used).
    episodes_file: str = os.getenv("EPISODES_FILE", "episodes_grpo_long.json")

    num_train_epochs: int = int(os.getenv("NUM_TRAIN_EPOCHS", "2"))
    per_device_train_batch_size: int = int(os.getenv("PER_DEVICE_TRAIN_BATCH_SIZE", "1"))
    gradient_accumulation_steps: int = int(os.getenv("GRADIENT_ACCUMULATION_STEPS", "1"))
    learning_rate: float = float(os.getenv("LEARNING_RATE", "5e-6"))
    # Bigger group => non-zero advantages even when one op dominates locally
    num_generations: int = int(os.getenv("NUM_GENERATIONS", "4"))
    beta: float = float(os.getenv("BETA", "0.04"))
    # Capacity 16 means memory list can grow long; bump prompt budget.
    max_prompt_length: int = int(os.getenv("MAX_PROMPT_LENGTH", "768"))
    max_completion_length: int = int(os.getenv("MAX_COMPLETION_LENGTH", "32"))
    # Sampling diversity for exploration of `remove` and `remove_index`.
    temperature: float = float(os.getenv("GRPO_TEMPERATURE", "1.3"))
    top_p: float = float(os.getenv("GRPO_TOP_P", "0.95"))
    top_k: int = int(os.getenv("GRPO_TOP_K", "50"))

    # Memory capacity used for long-horizon training. Pushed into env via
    # LONG_HORIZON_MEMORY_CAPACITY before the env class is imported (the env
    # reads this at class-definition time, so we set it as an env var here
    # if it has not already been set explicitly).
    memory_capacity: int = int(os.getenv("LONG_HORIZON_MEMORY_CAPACITY", "16"))

    dataset_size: int = int(os.getenv("DATASET_SIZE", "512"))
    rollouts_per_episode: int = int(os.getenv("ROLLOUTS_PER_EPISODE", "4"))
    seed: int = int(os.getenv("GRPO_SEED", "20260426"))

    # Reward shaping knobs for diversity / exploration during GRPO.
    diversity_bonus: float = float(os.getenv("GRPO_DIVERSITY_BONUS", "0.05"))
    diversity_penalty: float = float(os.getenv("GRPO_DIVERSITY_PENALTY", "0.20"))

    # Logit bias on the "remove" token at generation time. The SFT model
    # assigns ~0.5% probability to the remove operation; without this bias
    # GRPO never sees a reward signal for remove. After temperature warp,
    # +3.5 lifts P(remove) from ~0.5% to ~10-15%, so in a group of 4-6
    # generations there is typically at least one remove for advantage
    # estimation. The bias is small enough that grammar tokens elsewhere
    # in the JSON keep dominating; parse rate stays >95% in practice.
    remove_logit_bias: float = float(os.getenv("GRPO_REMOVE_LOGIT_BIAS", "3.5"))

    # Extra reward bump when the policy correctly chooses `remove`. This
    # accelerates the credit assignment that "remove of irrelevant slots is
    # good", offsetting the SFT bias against ever sampling remove at all.
    correct_remove_bonus: float = float(os.getenv("GRPO_CORRECT_REMOVE_BONUS", "0.20"))

    # Logging
    reward_log_file: str = os.getenv("REWARD_LOG_FILE", str(HERE / "grpo_reward_log.jsonl"))
    print_each_generation: bool = os.getenv("PRINT_EACH_GENERATION", "1") == "1"
    print_summary_every: int = int(os.getenv("PRINT_SUMMARY_EVERY", "20"))
    max_logged_text_chars: int = int(os.getenv("MAX_LOGGED_TEXT_CHARS", "300"))


CFG = TrainConfig()


def use_episodes(env_dir: Path, episodes_filename: str) -> Optional[Path]:
    env_dir = Path(env_dir)
    src = env_dir / episodes_filename
    if not src.exists():
        print(f"[grpo] episodes file {src} missing; using server/episodes.json as-is")
        return None

    dst = env_dir / "episodes.json"
    backup = env_dir / ".episodes_backup_for_grpo.json"
    if not backup.exists():
        backup.write_bytes(dst.read_bytes())
    dst.write_bytes(src.read_bytes())
    return backup


def restore_episodes(backup: Optional[Path], env_dir: Path) -> None:
    env_dir = Path(env_dir)
    if backup is None or not backup.exists():
        return
    dst = env_dir / "episodes.json"
    dst.write_bytes(backup.read_bytes())
    backup.unlink()


def to_obs_dict(env: LongHorizonMemoryEnvironment) -> Dict[str, Any]:
    return {
        "domain": env.current_domain,
        "task_name": env.current_difficulty,
        "memory": [{"index": i, "text": m.get("text", "")} for i, m in enumerate(env.memory)],
        "new_message": (env._current_message() or {}).get("text", ""),
    }


def to_score_state(env: LongHorizonMemoryEnvironment) -> Dict[str, Any]:
    msg = env._current_message()
    is_last = env.total_message_number == len(env.messages) - 1
    return {
        "memory": [
            {"text": m.get("text", ""), "isRelevant": bool(m.get("isRelevant", False))}
            for m in env.memory
        ],
        "message": None if msg is None else {
            "text": msg.get("text", ""),
            "isRelevant": bool(msg.get("isRelevant", False)),
        },
        "total_relevant_seen": env.total_relevant_seen,
        "is_last_step": is_last,
    }


def build_chat_prompt(observation_dict: Dict[str, Any], tokenizer) -> str:
    body = format_observation(observation_dict)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": body},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{body}\n<|assistant|>\n"


def _record_state(rows, env, fill_hist, op_hist, tokenizer) -> None:
    obs = to_obs_dict(env)
    score_state = to_score_state(env)
    prompt = build_chat_prompt(obs, tokenizer)
    rows.append({
        "prompt": prompt,
        "state_json": json.dumps(score_state, ensure_ascii=False),
    })
    fill_hist[len(env.memory)] += 1
    msg = env._current_message() or {}
    op_hist[("rel" if msg.get("isRelevant") else "irrel")] += 1


def _force_fill_to(env, target_fill: int, accept_irrelevant_prob: float, rng) -> None:
    """
    Drive the env forward by force-adding messages until memory has at least
    ``target_fill`` items or the episode ends. We always add relevant messages,
    and add irrelevant ones with probability ``accept_irrelevant_prob`` so the
    final memory has a controllable mix of relevant / irrelevant slots.
    """
    while not env._done and len(env.memory) < target_fill:
        msg = env._current_message() or {}
        is_rel = bool(msg.get("isRelevant", False))
        if is_rel or rng.random() < accept_irrelevant_prob:
            env.step(LongHorizonMemoryAction(operation="add"))
        else:
            env.step(LongHorizonMemoryAction(operation="noop"))


def build_dataset(tokenizer) -> Dataset:
    """
    Build a curated training set that mixes three kinds of (state, prompt)
    pairs so GRPO sees:

      1. ``oracle_natural`` (35%) - states reached by following the oracle.
         Distributionally close to what a competent policy will see.
      2. ``fill_first``    (35%) - states where memory is at or near capacity,
         forcing the policy to choose between ``add`` (rejected when full),
         ``remove`` (correct when an irrelevant slot exists), or ``noop``.
      3. ``remove_friendly`` (30%) - states where memory holds several
         irrelevant items, so a literal ``remove`` is the highest-reward op.

    A memory-fullness histogram is printed so you can see that the trainer
    is actually exercising long-horizon eviction (slots in the 8-16 range).
    """
    rng = random.Random(CFG.seed)
    capacity = LongHorizonMemoryEnvironment.MEMORY_CAPACITY
    env = LongHorizonMemoryEnvironment()

    rows: List[Dict[str, Any]] = []
    fill_hist: Counter = Counter()
    op_hist: Counter = Counter()
    style_hist: Counter = Counter()

    near_full = max(2, capacity - 2)

    while len(rows) < CFG.dataset_size:
        env.reset()
        style = rng.choices(
            ["oracle_natural", "fill_first", "remove_friendly"],
            weights=[0.20, 0.35, 0.45],
            k=1,
        )[0]
        style_hist[style] += 1

        if style == "fill_first":
            # Force memory to capacity with a heavy mix of irrelevants so that
            # subsequent steps are dominated by add-rejected-when-full and
            # remove-of-irrelevant decisions (where remove is uniquely best).
            _force_fill_to(env, capacity, accept_irrelevant_prob=0.7, rng=rng)
        elif style == "remove_friendly":
            # Pre-load 4-14 irrelevants. Now most steps are: agent sees a new
            # message while memory holds clear noise; remove of an irrelevant
            # slot is the highest-reward action.
            _force_fill_to(env, near_full, accept_irrelevant_prob=0.9, rng=rng)

        steps_in_ep = 0
        max_steps = max(8, CFG.rollouts_per_episode * 8)
        while not env._done and len(rows) < CFG.dataset_size and steps_in_ep < max_steps:
            _record_state(rows, env, fill_hist, op_hist, tokenizer)

            roll = rng.random()
            if roll < 0.4:
                action = env._oracle_action()
            elif roll < 0.65:
                action = LongHorizonMemoryAction(operation="add")
            elif roll < 0.85 and env.memory:
                action = LongHorizonMemoryAction(
                    operation="remove",
                    remove_index=rng.randrange(len(env.memory)),
                )
            else:
                action = LongHorizonMemoryAction(operation="noop")
            env.step(action)
            steps_in_ep += 1

    rng.shuffle(rows)
    rows = rows[: CFG.dataset_size]

    n = sum(fill_hist.values())
    if n > 0:
        print("[grpo] dataset memory utilization (slots filled when state was recorded):")
        for lo, hi, label in [
            (0, 0, "empty"),
            (1, 3, "1-3"),
            (4, 7, "4-7"),
            (8, 15, "8-15  (long-horizon)"),
            (16, 999, "16+   (full / capacity)"),
        ]:
            count = sum(c for k, c in fill_hist.items() if lo <= k <= hi)
            print(f"   {label:24s}: {count:5d} ({count / n:.1%})")
        print(f"[grpo] curated style mix: {dict(style_hist)}")
        print(f"[grpo] current-msg label mix in states: {dict(op_hist)}")
        print(f"[grpo] capacity={capacity}  episodes_file={CFG.episodes_file}")

    return Dataset.from_list(rows)


_JSON_OBJ_RE = re.compile(r"\{[^{}]*\}", re.DOTALL)


def _extract_first_json(text: str) -> Optional[str]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
        text = re.sub(r"\s*```\s*$", "", text)
    m = _JSON_OBJ_RE.search(text)
    return m.group(0) if m else None


def _completion_text(c: Any) -> str:
    if isinstance(c, str):
        return c
    if isinstance(c, list) and c:
        last = c[-1]
        if isinstance(last, dict) and "content" in last:
            return str(last["content"])
    return str(c)


def _parse_action(text: str) -> Tuple[Optional[LongHorizonMemoryAction], str]:
    obj_str = _extract_first_json(text)
    if obj_str is None:
        return None, "no_json"
    try:
        obj = json.loads(obj_str)
    except json.JSONDecodeError:
        return None, "json_invalid"
    try:
        return LongHorizonMemoryAction.model_validate(obj), "ok"
    except Exception:
        return None, "pydantic_invalid"


class RewardMonitor:
    def __init__(self, log_file: str, print_each_generation: bool, print_summary_every: int, max_chars: int):
        self.log_file = Path(log_file)
        self.log_file.parent.mkdir(parents=True, exist_ok=True)
        self.print_each_generation = print_each_generation
        self.print_summary_every = max(1, print_summary_every)
        self.max_chars = max_chars

        self.call_idx = 0
        self.total_samples = 0
        self.total_reward_sum = 0.0
        self.task_reward_sum = 0.0
        self.format_reward_sum = 0.0

        self.parse_status = Counter()
        self.action_ops = Counter()

    def _clip(self, x: str) -> str:
        x = str(x) if x is not None else ""
        return x if len(x) <= self.max_chars else (x[: self.max_chars] + "...<truncated>")

    def log_one(
        self,
        sample_idx: int,
        completion_text: str,
        parse_status: str,
        action_obj: Optional[LongHorizonMemoryAction],
        task_reward: float,
        format_reward: float,
        total_reward: float,
        prompt_text: Optional[str] = None,
    ):
        self.total_samples += 1
        self.total_reward_sum += total_reward
        self.task_reward_sum += task_reward
        self.format_reward_sum += format_reward
        self.parse_status[parse_status] += 1

        op = None
        remove_idx = None
        if action_obj is not None:
            op = action_obj.operation
            remove_idx = action_obj.remove_index
            self.action_ops[op] += 1

        record = {
            "ts": datetime.utcnow().isoformat() + "Z",
            "reward_call_idx": self.call_idx,
            "sample_idx": sample_idx,
            "parse_status": parse_status,
            "operation": op,
            "remove_index": remove_idx,
            "task_reward": float(task_reward),
            "format_reward": float(format_reward),
            "total_reward": float(total_reward),
            "completion_text": self._clip(completion_text),
            "prompt_text": self._clip(prompt_text or ""),
        }
        with self.log_file.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        if self.print_each_generation:
            print(
                f"[reward] call={self.call_idx} sample={sample_idx} "
                f"status={parse_status} op={op} "
                f"task={task_reward:+.3f} fmt={format_reward:+.3f} total={total_reward:+.3f}"
            )

    def maybe_print_summary(self):
        if self.call_idx % self.print_summary_every != 0:
            return

        n = max(1, self.total_samples)
        parse_ok = self.parse_status["ok"] / n
        mean_total = self.total_reward_sum / n
        mean_task = self.task_reward_sum / n
        mean_fmt = self.format_reward_sum / n

        action_total = sum(self.action_ops.values())
        add_rate = self.action_ops["add"] / action_total if action_total else 0.0
        rem_rate = self.action_ops["remove"] / action_total if action_total else 0.0
        noop_rate = self.action_ops["noop"] / action_total if action_total else 0.0

        print(
            "[reward-summary] "
            f"calls={self.call_idx} samples={n} "
            f"parse_ok={parse_ok:.2%} "
            f"mean_total={mean_total:+.4f} mean_task={mean_task:+.4f} mean_fmt={mean_fmt:+.4f} "
            f"ops(add/remove/noop)=({add_rate:.1%}/{rem_rate:.1%}/{noop_rate:.1%}) "
            f"fail(no_json/json_invalid/pydantic_invalid)="
            f"({self.parse_status['no_json']}/{self.parse_status['json_invalid']}/{self.parse_status['pydantic_invalid']})"
        )


REWARD_MONITOR = RewardMonitor(
    log_file=CFG.reward_log_file,
    print_each_generation=CFG.print_each_generation,
    print_summary_every=CFG.print_summary_every,
    max_chars=CFG.max_logged_text_chars,
)


def combined_reward_fn(completions, **kwargs) -> List[float]:
    """
    Compute task + format + diversity rewards.

    The diversity term is what fixes the noop-collapse / advantage-zero issue:
    if every completion in a GRPO group picks the same operation, we apply a
    small penalty (so the group is not stuck at zero advantage); if the group
    samples >=3 distinct ops we apply a small bonus to encourage exploration.
    The diversity term is small relative to ``score_action`` so it cannot
    drown out the actual task signal.
    """
    states = kwargs.get("state_json")
    prompts = kwargs.get("prompts", None) or kwargs.get("prompt", None)

    if states is None:
        return [0.0] * len(completions)

    capacity = LongHorizonMemoryEnvironment.MEMORY_CAPACITY
    REWARD_MONITOR.call_idx += 1

    parsed: List[Tuple[str, Optional[LongHorizonMemoryAction], str]] = []
    for comp in completions:
        text = _completion_text(comp)
        action, status = _parse_action(text)
        parsed.append((text, action, status))

    group_members: Dict[str, List[int]] = defaultdict(list)
    for i, raw_state in enumerate(states):
        state_key = raw_state if isinstance(raw_state, str) else json.dumps(raw_state, sort_keys=True)
        group_members[state_key].append(i)

    diversity_reward = [0.0] * len(completions)
    for _, members in group_members.items():
        if len(members) < 2:
            continue
        ops = [parsed[i][1].operation if parsed[i][1] is not None else None for i in members]
        unique = {o for o in ops if o is not None}
        if len(unique) <= 1:
            for i in members:
                diversity_reward[i] -= CFG.diversity_penalty
        elif len(unique) >= 3:
            for i in members:
                diversity_reward[i] += CFG.diversity_bonus

    rewards: List[float] = []
    for i, (raw_state, (completion_text, action, status)) in enumerate(zip(states, parsed)):
        state = json.loads(raw_state) if isinstance(raw_state, str) else raw_state

        if action is None:
            task_r = -1.0
        else:
            try:
                task_r, _ = score_action(
                    state,
                    action.model_dump(exclude_none=True),
                    capacity=capacity,
                )
                task_r = float(task_r)
            except Exception:
                task_r = -1.0

        # Explicit "correct remove" bonus. When the policy rolls a `remove`
        # that the env scored positively (REWARD_REMOVE_IRRELEVANT == +0.4),
        # we add an extra bonus so the gradient toward `remove` is at least
        # as large as the gradient toward `add` on a relevant message. This
        # offsets the SFT bias against the remove token.
        remove_bonus = 0.0
        if action is not None and action.operation == "remove" and task_r > 0.0:
            remove_bonus = CFG.correct_remove_bonus

        fmt_r = 0.05 if action is not None else -0.1
        div_r = diversity_reward[i]
        total_r = float(task_r + fmt_r + div_r + remove_bonus)
        rewards.append(total_r)

        prompt_text = None
        if isinstance(prompts, list) and i < len(prompts):
            prompt_text = prompts[i]

        REWARD_MONITOR.log_one(
            sample_idx=i,
            completion_text=completion_text,
            parse_status=status,
            action_obj=action,
            task_reward=task_r + remove_bonus,
            format_reward=fmt_r + div_r,
            total_reward=total_r,
            prompt_text=prompt_text,
        )

    REWARD_MONITOR.maybe_print_summary()
    return rewards


class _AdditiveTokenLogitBias(LogitsProcessor):
    """Add a small additive bias to a fixed set of token ids during generation.

    This is the exploration mechanism that lets GRPO discover the `remove`
    operation. Without it, the SFT policy assigns ~1e-3 probability to the
    `remove` token, so even at temperature 1.3 it is sampled roughly never;
    GRPO can only learn from actions that get sampled at least sometimes.
    """

    def __init__(self, token_ids: List[int], bias: float):
        super().__init__()
        self.token_ids = list(token_ids)
        self.bias = float(bias)

    def __call__(self, input_ids, scores):
        if not self.token_ids or self.bias == 0.0:
            return scores
        scores[:, self.token_ids] = scores[:, self.token_ids] + self.bias
        return scores


def _resolve_remove_token_ids(tokenizer) -> List[int]:
    """Find the token ids that BPE produces for the word ``remove`` in JSON.

    JSON action output is shaped like ``{"operation":"remove",...}``. After the
    quote, BPE typically encodes ``remove`` as a single token; we also try a
    few neighboring contexts (leading space, leading quote) in case the model
    sometimes emits them.
    """
    candidates: List[str] = ["remove", " remove", '"remove']
    out: List[int] = []
    seen = set()
    for cand in candidates:
        try:
            ids = tokenizer.encode(cand, add_special_tokens=False)
        except Exception:
            continue
        if not ids:
            continue
        for tok_id in ids:
            try:
                decoded = tokenizer.decode([tok_id]).strip().strip('"').lower()
            except Exception:
                decoded = ""
            if decoded.startswith("remov") and tok_id not in seen:
                out.append(tok_id)
                seen.add(tok_id)
    return out


def _attach_remove_logit_bias(model, tokenizer, bias: float) -> List[int]:
    """Monkey-patch ``model.generate`` to always include the remove-bias
    processor. Returns the list of biased token ids for logging."""
    if bias == 0.0:
        return []
    token_ids = _resolve_remove_token_ids(tokenizer)
    if not token_ids:
        print("[grpo] WARNING: could not resolve `remove` token ids; skipping logit bias")
        return []

    processor = _AdditiveTokenLogitBias(token_ids=token_ids, bias=bias)

    original_generate = model.generate

    def biased_generate(*args, **kwargs):
        existing = kwargs.get("logits_processor")
        if existing is None:
            kwargs["logits_processor"] = LogitsProcessorList([processor])
        else:
            existing_list = list(existing)
            if processor not in existing_list:
                existing_list.append(processor)
            kwargs["logits_processor"] = LogitsProcessorList(existing_list)
        return original_generate(*args, **kwargs)

    model.generate = biased_generate  # type: ignore[assignment]
    return token_ids


def load_model_and_tokenizer():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required to fine-tune Qwen 2.5-1.5B with 4-bit quantization.")

    print(f"[grpo] Loading tokenizer: {CFG.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    print(f"[grpo] Loading 4-bit base model: {CFG.model_name}")
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    base = AutoModelForCausalLM.from_pretrained(
        CFG.model_name,
        trust_remote_code=True,
        quantization_config=bnb,
        device_map={"": 0},  # single GPU for notebook stability
    )
    base.config.use_cache = False

    print(f"[grpo] Attaching SFT LoRA adapter (trainable=True): {CFG.adapter_path}")
    model = PeftModel.from_pretrained(base, CFG.adapter_path, is_trainable=True)
    model.gradient_checkpointing_enable()
    model.print_trainable_parameters()
    return model, tokenizer


def main() -> None:
    env_dir = HERE / "server"
    backup = use_episodes(env_dir, CFG.episodes_file)
    try:
        try:
            from trl import GRPOConfig, GRPOTrainer
        except ImportError as e:
            raise ImportError(
                "[grpo] Missing dependency 'trl'. Run: %pip install -U trl accelerate peft"
            ) from e

        random.seed(CFG.seed)
        torch.manual_seed(CFG.seed)

        print(
            "[grpo] CONFIG  "
            f"capacity={LongHorizonMemoryEnvironment.MEMORY_CAPACITY} "
            f"episodes={CFG.episodes_file}  "
            f"num_generations={CFG.num_generations} "
            f"temperature={CFG.temperature} top_p={CFG.top_p} top_k={CFG.top_k}  "
            f"max_prompt={CFG.max_prompt_length} max_compl={CFG.max_completion_length}  "
            f"dataset_size={CFG.dataset_size}"
        )

        model, tokenizer = load_model_and_tokenizer()

        biased_ids = _attach_remove_logit_bias(model, tokenizer, CFG.remove_logit_bias)
        if biased_ids:
            print(
                f"[grpo] Logit-biasing remove token ids {biased_ids} "
                f"by +{CFG.remove_logit_bias} (enables exploration of the remove op)"
            )

        print(f"[grpo] Building dataset of {CFG.dataset_size} states from {CFG.episodes_file}")
        dataset = build_dataset(tokenizer)
        print(f"[grpo] Dataset built. Columns: {list(dataset.column_names)}")
        print(f"[grpo] Reward log file: {CFG.reward_log_file}")

        grpo_config_kwargs: Dict[str, Any] = dict(
            output_dir=CFG.output_dir,
            num_train_epochs=CFG.num_train_epochs,
            per_device_train_batch_size=CFG.per_device_train_batch_size,
            gradient_accumulation_steps=CFG.gradient_accumulation_steps,
            learning_rate=CFG.learning_rate,
            logging_steps=5,
            save_strategy="epoch",
            report_to="none",
            num_generations=CFG.num_generations,
            generation_batch_size=CFG.num_generations,  # ensures divisibility
            beta=CFG.beta,
            max_prompt_length=CFG.max_prompt_length,
            max_completion_length=CFG.max_completion_length,
            temperature=CFG.temperature,
            top_p=CFG.top_p,
            top_k=CFG.top_k,
            seed=CFG.seed,
            bf16=False,
            fp16=False,
            optim="paged_adamw_8bit",
            max_grad_norm=1.0,
            remove_unused_columns=False,
            dataloader_num_workers=0,
            log_completions=True,
            num_completions_to_print=2,
        )

        valid = set(GRPOConfig.__init__.__code__.co_varnames)
        grpo_config_kwargs = {k: v for k, v in grpo_config_kwargs.items() if k in valid}
        args = GRPOConfig(**grpo_config_kwargs)

        trainer = GRPOTrainer(
            model=model,
            args=args,
            train_dataset=dataset,
            reward_funcs=[combined_reward_fn],
            processing_class=tokenizer,
        )

        print("[grpo] Starting training...")
        trainer.train()
        trainer.save_model(CFG.output_dir)
        tokenizer.save_pretrained(CFG.output_dir)
        print(f"[grpo] Saved adapter and tokenizer to: {CFG.output_dir}")
    finally:
        restore_episodes(backup, env_dir)


# Notebook usage:
#   import train_grpo_memory
#   train_grpo_memory.main()
#
# Auto-run only when the file is the entry point. This guard keeps the
# module importable for tests and inspection without launching training.
if __name__ == "__main__":
    main()

[grpo] CONFIG  capacity=16 episodes=episodes_grpo_long.json  num_generations=2 temperature=1.3 top_p=0.95 top_k=50  max_prompt=768 max_compl=24  dataset_size=256
[grpo] Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct
[grpo] Loading 4-bit base model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[grpo] Attaching SFT LoRA adapter (trainable=True): /kaggle/working/open_env_meta_torch_comp/memory_action_sft_qwen15b
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
[grpo] Logit-biasing remove token ids [5399, 4057] by +3.5 (enables exploration of the remove op)
[grpo] Building dataset of 256 states from episodes_grpo_long.json
[grpo] dataset memory utilization (slots filled when state was recorded):
   empty                   :    18 (7.0%)
   1-3                     :    14 (5.5%)
   4-7                     :    13 (5.1%)
   8-15  (long-horizon)    :   196 (76.6%)
   16+   (full / capacity) :    15 (5.9%)
[grpo] curated style mix: {'fill_first': 3, 'remove_friendly': 4, 'oracle_natural': 1}
[grpo] current-msg label mix in states: {'irrel': 198, 'rel': 58}
[grpo] capacity=16  episodes_file=episodes_grpo_long.json
[grpo] Dataset built. Columns: ['prompt', 'state_json']
[grpo] Reward log file: /kaggle/working/open_env_meta_torch_comp/grpo_reward_log.json

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


[grpo] Starting training...


...

╭──────────────────────────────────────────────────── Step 90 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                              ┃ Completion           ┃ combined_reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                              │ {"operation": "add"} │               0.45 │      0.00 │ │
│ │ You are a memory manager. Decide exactly one action │                      │                    │           │ │
│ │ for the new message.                                │                      │                    │           │ │
│ │ Return ONLY one JSON object with one of these       │                      │                    │           │ │
│ │ forms:                                              │                      │                    │           │ │
│ │ {"operation":"add"}                                 │                      │                    │           │ │
│ │ {"operation":"noop"}                                │                      │                    │           │ │
│ │ {"operation":"remove","remove_index":<int>}         │                      │                    │           │ │
│ │ Do not output markdown, explanations, or extra      │                      │                    │           │ │
│ │ text.                                               │                      │                    │           │ │
│ │ user                                                │                      │                    │           │ │
│ │ Domain: machine_learning_ops                        │                      │                    │           │ │
│ │ Task difficulty: hard                               │                      │                    │           │ │
│ │ Memory entries (9):                                 │                      │                    │           │ │
│ │ [0] The model registry drift alerts fire on benign  │                      │                    │           │ │
│ │ weekend traffic, and validation loss diverges in    │                      │                    │           │ │
│ │ the third epoch.                                    │                      │                    │           │ │
│ │ [1] The model registry online model latency creeps  │                      │                    │           │ │
│ │ up over a day, and feature freshness lag peaks at   │                      │                    │           │ │
│ │ 18 minutes.                                         │                      │                    │           │ │
│ │ [2] The model registry the autoscaler oscillates    │                      │                    │           │ │
│ │ between min and max replicas, and feature freshness │                      │                    │           │ │
│ │ lag peaks at 18 minutes.                            │                      │                    │           │ │
│ │ [3] The embedding refresh job training fails after  │                      │                    │           │ │
│ │ the data validation step because feature freshness  │                      │                    │           │ │
│ │ lag peaks at 18 minutes.                            │                      │                    │           │ │
│ │ [4] The model registry shadow models silently       │                      │                    │           │ │
│ │ diverge from production because logits drift slowly │                      │                    │           │ │
│ │ on the long tail segment.                           │                      │                    │           │ │
│ │ [5] We will add a circuit breaker around the slow   │                      │                    │           │ │
│ │ feature after we noted that p99 latency aligns with 

[reward] call=46 sample=0 status=ok op=add task=-0.600 fmt=-0.150 total=-0.750
[reward] call=46 sample=1 status=ok op=add task=-0.600 fmt=-0.150 total=-0.750
[reward] call=47 sample=0 status=ok op=noop task=+0.050 fmt=-0.150 total=-0.100
[reward] call=47 sample=1 status=ok op=noop task=+0.050 fmt=-0.150 total=-0.100
[reward] call=48 sample=0 status=ok op=noop task=+0.050 fmt=-0.150 total=-0.100
[reward] call=48 sample=1 status=ok op=noop task=+0.050 fmt=-0.150 total=-0.100


╭──────────────────────────────────────────────────── Step 95 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                             ┃ Completion            ┃ combined_reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                             │ {"operation": "noop"} │              -0.10 │      0.00 │ │
│ │ You are a memory manager. Decide exactly one       │                       │                    │           │ │
│ │ action for the new message.                        │                       │                    │           │ │
│ │ Return ONLY one JSON object with one of these      │                       │                    │           │ │
│ │ forms:                                             │                       │                    │           │ │
│ │ {"operation":"add"}                                │                       │                    │           │ │
│ │ {"operation":"noop"}                               │                       │                    │           │ │
│ │ {"operation":"remove","remove_index":<int>}        │                       │                    │           │ │
│ │ Do not output markdown, explanations, or extra     │                       │                    │           │ │
│ │ text.                                              │                       │                    │           │ │
│ │ user                                               │                       │                    │           │ │
│ │ Domain: game_engine_optimization                   │                       │                    │           │ │
│ │ Task difficulty: hard                              │                       │                    │           │ │
│ │ Memory entries (13):                               │                       │                    │           │ │
│ │ [0] The AI navigation mesh the physics solver      │                       │                    │           │ │
│ │ under-resolves rapid contacts.                     │                       │                    │           │ │
│ │ [1] The input handling layer frame time fluctuates │                       │                    │           │ │
│ │ on dense outdoor scenes because GPU bound passes   │                       │                    │           │ │
│ │ peak in the foliage area.                          │                       │                    │           │ │
│ │ [2] My favorite tea shop discontinued an oolong I  │                       │                    │           │ │
│ │ liked.                                             │                       │                    │           │ │
│ │ [3] The AI navigation mesh asset streaming hitches │                       │                    │           │ │
│ │ during fast travel, and audio clip events cluster  │                       │                    │           │ │
│ │ around boss intros.                                │                       │                    │           │ │
│ │ [4] We need memory budgets must respect platform   │                       │                    │           │ │
│ │ constraints, but input lag exceeds target on       │                       │                    │           │ │
│ │ certain controllers.                               │                       │                    │           │ │
│ │ [5] We will introduce continuous collision         │                       │                    │           │ │
│ │ detection so we can address asset streaming        │                       │                    │           │ │
│ │ hitches during fast travel.                        │                       │                    │           │ │
│ │ [6] The AI navigation mesh input lag exceeds       │

[reward] call=49 sample=0 status=ok op=add task=+0.600 fmt=+0.050 total=+0.650
[reward] call=49 sample=1 status=ok op=remove task=-0.500 fmt=+0.050 total=-0.450
[reward] call=50 sample=0 status=ok op=remove task=-0.500 fmt=-0.150 total=-0.650
[reward] call=50 sample=1 status=ok op=remove task=-0.500 fmt=-0.150 total=-0.650


KeyboardInterrupt: 

In [9]:
!zip -r /kaggle/working/memory_action_grpo_qwen15b.zip /kaggle/working/open_env_meta_torch_comp/memory_action_grpo_qwen15b

  adding: kaggle/working/open_env_meta_torch_comp/memory_action_grpo_qwen15b/ (stored 0%)
  adding: kaggle/working/open_env_meta_torch_comp/memory_action_grpo_qwen15b/adapter_model.safetensors (deflated 22%)
  adding: kaggle/working/open_env_meta_torch_comp/memory_action_grpo_qwen15b/completions/ (stored 0%)
  adding: kaggle/working/open_env_meta_torch_comp/memory_action_grpo_qwen15b/completions/completions_00770.parquet (deflated 75%)
  adding: kaggle/working/open_env_meta_torch_comp/memory_action_grpo_qwen15b/completions/completions_00290.parquet (deflated 73%)
  adding: kaggle/working/open_env_meta_torch_comp/memory_action_grpo_qwen15b/completions/completions_00200.parquet (deflated 75%)
  adding: kaggle/working/open_env_meta_torch_comp/memory_action_grpo_qwen15b/completions/completions_00650.parquet (deflated 74%)
  adding: kaggle/working/open_env_meta_torch_comp/memory_action_grpo_qwen15b/completions/completions_00675.parquet (deflated 69%)
  adding: kaggle/working/open_env_meta_t

In [7]:
ls

benchmark_models.py      __init__.py                  random_baseline.py
benchmark_results/       kaggle_benchmarking/         README.md
client.py                memory_action_grpo_qwen15b/  server/
data.py                  memory_action_sft_qwen15b/   train_grpo_memory.py
deployment_commands.txt  models.py                    train_sft_qwen.py
Dockerfile               openenv.yaml                 uv.lock
grpo_reward_log.jsonl    __pycache__/                 validation.txt
inference.py             pyproject.toml               verify_sft_model.py


In [12]:
! python benchmark_models.py

[bench] writing results to /kaggle/working/open_env_meta_torch_comp/benchmark_results/run_20260426_044001
[bench] selected 20 episodes: [146, 355, 326, 247, 193, 116, 357, 319, 159, 101]...

[bench] === base_1.5b ===  (Qwen2.5-1.5B-Instruct, no fine-tuning. Floor for the action format.)
[bench] [base_1.5b] loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct
[bench] [base_1.5b] loading 4-bit base: Qwen/Qwen2.5-1.5B-Instruct
model.safetensors:  48%|█████████▋          | 1.50G/3.09G [00:13<00:14, 108MB/s]
Loading weights: 100%|█| 338/338 [00:01<00:00, 295.90it/s, Materializing param=m
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 1.05MB/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[bench] [base_1.5b] round=10 active=20/20 elapsed=26.7s
[bench] [base_1.5b] round=20 active=20/20 elapsed=60.5s
[bench] [base_1.5b] round=30 active=20/20 elapsed=99.3s
[bench] [base_1.

In [ ]:
"""
Benchmark four memory-action policies on the Long Horizon Memory env.

Models compared (configurable via env vars; see BENCH_LLM_BASE and BENCH_ALT_HF):

    base_1.5b   - Qwen/Qwen2.5-1.5B-Instruct, no LoRA
    sft_1.5b    - Qwen/Qwen2.5-1.5B-Instruct + memory_action_sft_qwen15b
    grpo_1.5b   - Qwen/Qwen2.5-1.5B-Instruct + memory_action_grpo_qwen15b
    (fourth)    - Larger untrained instruct baseline, default: Qwen2.5-3B (4-bit
                  NF4 + double quant). Use BENCH_LLM_BASE=0.5b|3b|7b or
                  BENCH_ALT_HF to avoid OOM (7B is heavy even in 4-bit).

Why this script exists
----------------------
After SFT and GRPO finish we want to *measure* whether RL actually helped, on
the same eval set, with the same prompt format, and the same decoding rules.
Comparing four models in one harness lets us isolate:
  - the value of SFT alone over a base instruct model
  - the marginal value of GRPO over SFT
  - whether a 5x bigger general-purpose model beats a 1.5B model that has been
    trained specifically for the task.

Outputs (under ./benchmark_results/<run_id>/):
  - <model_id>_steps.jsonl       per-step records (one JSON per generation)
  - <model_id>_episodes.jsonl    per-episode aggregates
  - <model_id>_summary.json      aggregated metrics across all episodes
  - comparison.json              all summaries side by side
  - comparison.md                human-readable report incl. failure modes
                                 and per-episode head-to-head highlights
  - comparison_table.txt         ASCII table printed at the end

Run from the project root:

    cd Long_Horizon_Memory
    python benchmark_models.py

Useful overrides:

    N_EPISODES=15
    BENCH_LLM_BASE=3b                 (0.5b|3b|7b, or BENCH_ALT_HF for a custom model)
    BATCH_SIZE=4                      (1.5B+LoRA; lower if you still OOM)
    BATCH_SIZE_ALT=1                  (larger baseline; 1 = safest)
    BENCH_4BIT_COMPUTE=bfloat16       (or float16; bf16 is higher-quality 4-bit matmuls)
    BENCH_4BIT_DOUBLE_QUANT=1         (1 saves VRAM; set 0 to disable)
    MAX_NEW_TOKENS=32
    SKIP_MODELS=base_3b               (comma-separated; also base_7b, base_05b)
    EPISODES_FILE=episodes_grpo_long.json
    LONG_HORIZON_MEMORY_CAPACITY=16
    PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True  (helps fragmentation)
"""

from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

# Capacity must be set before the env class is imported because the env reads
# LONG_HORIZON_MEMORY_CAPACITY at class-definition time.
os.environ.setdefault("LONG_HORIZON_MEMORY_CAPACITY", "16")

import torch  # noqa: E402


# ── path resolution (matches train_grpo_memory.py) ──────────────────────────


def _resolve_here() -> Path:
    if "__file__" in globals():
        p = Path(__file__).resolve().parent
        if (p / "server").exists() and (p / "data.py").exists():
            return p
    cwd = Path.cwd().resolve()
    if (cwd / "server").exists() and (cwd / "data.py").exists():
        return cwd
    if (cwd / "Long_Horizon_Memory" / "server").exists():
        return cwd / "Long_Horizon_Memory"
    raise RuntimeError("Could not locate Long_Horizon_Memory root.")


HERE = _resolve_here()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE / "server"))


from data import SYSTEM_PROMPT, format_observation  # noqa: E402
from models import LongHorizonMemoryAction  # noqa: E402
from server.long_horizon_memory_environment import (  # noqa: E402
    LongHorizonMemoryEnvironment,
)


# ── benchmark configuration ─────────────────────────────────────────────────


def _env_int(name: str, default: int) -> int:
    v = os.getenv(name, "")
    if v.strip():
        return int(v)
    return default


@dataclass
class BenchConfig:
    n_episodes: int = int(os.getenv("N_EPISODES", "20"))
    seed: int = int(os.getenv("BENCH_SEED", "20260427"))
    max_new_tokens: int = int(os.getenv("MAX_NEW_TOKENS", "32"))
    batch_size: int = _env_int("BATCH_SIZE", 4)
    # Larger untrained baseline (3B/7B/0.5B): use 1 to avoid OOM during batched generate.
    batch_size_alt: int = _env_int(
        "BATCH_SIZE_ALT",
        _env_int("BATCH_SIZE_7B", 1),
    )
    episodes_file: str = os.getenv("EPISODES_FILE", "episodes_grpo_long.json")
    capacity: int = int(os.getenv("LONG_HORIZON_MEMORY_CAPACITY", "16"))

    skip_models: List[str] = field(
        default_factory=lambda: [
            s.strip() for s in os.getenv("SKIP_MODELS", "").split(",") if s.strip()
        ]
    )
    only_models: List[str] = field(
        default_factory=lambda: [
            s.strip() for s in os.getenv("ONLY_MODELS", "").split(",") if s.strip()
        ]
    )

    output_root: str = os.getenv("OUTPUT_ROOT", str(HERE / "benchmark_results"))
    run_id: str = os.getenv("BENCH_RUN_ID", datetime.now().strftime("run_%Y%m%d_%H%M%S"))


CFG = BenchConfig()


@dataclass
class ModelSpec:
    model_id: str
    hf_name: str
    adapter_path: Optional[str]
    quant_4bit: bool = True
    description: str = ""
    # Use a small micro-batch for this run (bigger model or long prompts).
    use_low_vram_batch: bool = False


# Keys: choose with BENCH_LLM_BASE (default 3b — much lighter than 7B for 4-bit VRAM).
BENCH_ALT_REGISTRY: Dict[str, Tuple[str, str, str]] = {
    "0.5b": (
        "base_05b",
        "Qwen/Qwen2.5-0.5B-Instruct",
        "Untrained 0.5B instruct (smallest VRAM).",
    ),
    "3b": (
        "base_3b",
        "Qwen/Qwen2.5-3B-Instruct",
        "Untrained 3B instruct (default larger baseline; lighter than 7B in 4-bit).",
    ),
    "7b": (
        "base_7b",
        "Qwen/Qwen2.5-7B-Instruct",
        "Untrained 7B instruct (needs more VRAM; use BATCH_SIZE_ALT=1).",
    ),
}


def _alt_baseline_spec() -> ModelSpec:
    custom_hf = os.getenv("BENCH_ALT_HF", "").strip()
    custom_id = os.getenv("BENCH_ALT_MODEL_ID", "base_custom").strip() or "base_custom"
    if custom_hf:
        return ModelSpec(
            model_id=custom_id,
            hf_name=custom_hf,
            adapter_path=None,
            use_low_vram_batch=True,
            description=os.getenv(
                "BENCH_ALT_DESC",
                f"Custom larger baseline from BENCH_ALT_HF (no LoRA).",
            ),
        )
    key = os.getenv("BENCH_LLM_BASE", "3b").lower().strip()
    if key not in BENCH_ALT_REGISTRY:
        valid = ", ".join(sorted(BENCH_ALT_REGISTRY.keys()))
        raise ValueError(f"BENCH_LLM_BASE={key!r} must be one of: {valid}, or set BENCH_ALT_HF.")
    mid, hf, desc = BENCH_ALT_REGISTRY[key]
    return ModelSpec(
        model_id=mid,
        hf_name=hf,
        adapter_path=None,
        use_low_vram_batch=True,
        description=desc,
    )


def _default_model_specs() -> List[ModelSpec]:
    return [
        ModelSpec(
            model_id="base_1.5b",
            hf_name="Qwen/Qwen2.5-1.5B-Instruct",
            adapter_path=None,
            description="Qwen2.5-1.5B-Instruct, no fine-tuning. Floor for the action format.",
        ),
        ModelSpec(
            model_id="sft_1.5b",
            hf_name="Qwen/Qwen2.5-1.5B-Instruct",
            adapter_path=str(HERE / "memory_action_sft_qwen15b"),
            description="SFT only. Same base + memory_action_sft_qwen15b LoRA.",
        ),
        ModelSpec(
            model_id="grpo_1.5b",
            hf_name="Qwen/Qwen2.5-1.5B-Instruct",
            adapter_path=str(HERE / "memory_action_grpo_qwen15b"),
            description="GRPO continued from SFT. memory_action_grpo_qwen15b LoRA.",
        ),
        _alt_baseline_spec(),
    ]


MODEL_SPECS: List[ModelSpec] = _default_model_specs()


def selected_models() -> List[ModelSpec]:
    if CFG.only_models:
        wanted = set(CFG.only_models)
        return [m for m in MODEL_SPECS if m.model_id in wanted]
    skip = set(CFG.skip_models)
    return [m for m in MODEL_SPECS if m.model_id not in skip]


# ── prompt + completion helpers ─────────────────────────────────────────────


def to_obs_dict(env: LongHorizonMemoryEnvironment) -> Dict[str, Any]:
    return {
        "domain": env.current_domain,
        "task_name": env.current_difficulty,
        "memory": [{"index": i, "text": m.get("text", "")} for i, m in enumerate(env.memory)],
        "new_message": (env._current_message() or {}).get("text", ""),
    }


def build_chat_prompt(obs: Dict[str, Any], tokenizer) -> str:
    body = format_observation(obs)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": body},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{body}\n<|assistant|>\n"


_JSON_OBJ_RE = re.compile(r"\{[^{}]*\}", re.DOTALL)


def _extract_first_json(text: str) -> Optional[str]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
        text = re.sub(r"\s*```\s*$", "", text)
    m = _JSON_OBJ_RE.search(text)
    return m.group(0) if m else None


def parse_completion(text: str) -> Tuple[Optional[LongHorizonMemoryAction], str]:
    obj_str = _extract_first_json(text)
    if obj_str is None:
        return None, "no_json"
    try:
        obj = json.loads(obj_str)
    except json.JSONDecodeError:
        return None, "json_invalid"
    try:
        return LongHorizonMemoryAction.model_validate(obj), "ok"
    except Exception:
        return None, "pydantic_invalid"


def fallback_action() -> LongHorizonMemoryAction:
    """When parsing fails the env still needs *something* to step. We use noop
    so we don't punish the env state further; the reward is already low for
    the unparseable completion via ``score_action``."""
    return LongHorizonMemoryAction(operation="noop")


# ── episode-store swap (so the env reads the long-horizon dataset) ──────────


def use_episodes(env_dir: Path, episodes_filename: str) -> Optional[Path]:
    src = env_dir / episodes_filename
    if not src.exists():
        print(f"[bench] episodes file {src} missing; using server/episodes.json as-is")
        return None
    dst = env_dir / "episodes.json"
    backup = env_dir / ".episodes_backup_for_bench.json"
    if not backup.exists():
        backup.write_bytes(dst.read_bytes())
    dst.write_bytes(src.read_bytes())
    return backup


def restore_episodes(backup: Optional[Path], env_dir: Path) -> None:
    if backup is None or not backup.exists():
        return
    (env_dir / "episodes.json").write_bytes(backup.read_bytes())
    backup.unlink()


def select_eval_episode_ids(n: int, seed: int) -> List[int]:
    """Pick n distinct episode_ids deterministically from the loaded eps file.
    The same seed yields the same set across runs and across models."""
    env = LongHorizonMemoryEnvironment()
    all_ids = [int(ep.get("episode_id", i + 1)) for i, ep in enumerate(env.episodes)]
    rng = random.Random(seed)
    rng.shuffle(all_ids)
    return all_ids[: min(n, len(all_ids))]


# ── model loading ───────────────────────────────────────────────────────────


def _bitsandbytes_4bit_config():
    """NF4 4-bit weights with optional double quant (saves VRAM) and high-quality
    compute dtype for the matmuls (bf16 or fp16, via BENCH_4BIT_COMPUTE)."""
    from transformers import BitsAndBytesConfig

    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    dq = os.getenv("BENCH_4BIT_DOUBLE_QUANT", "1").strip().lower() not in (
        "0",
        "false",
        "no",
        "off",
    )
    qtype = os.getenv("BENCH_4BIT_TYPE", "nf4").strip().lower()
    if qtype not in ("nf4", "fp4"):
        qtype = "nf4"
    comp = os.getenv("BENCH_4BIT_COMPUTE", "bfloat16").strip().lower()
    if comp in ("bf16", "bfloat16"):
        compute_dtype = torch.bfloat16
    else:
        compute_dtype = torch.float16
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=dq,
        bnb_4bit_quant_type=qtype,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    return bnb


def load_model_and_tokenizer(spec: ModelSpec):
    """Load a Qwen base + optional LoRA adapter in 4-bit on a single GPU."""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required to run the benchmark in 4-bit.")

    bnb = _bitsandbytes_4bit_config()
    print(
        f"[bench] 4bit: type={bnb.bnb_4bit_quant_type} double_quant={bnb.bnb_4bit_use_double_quant} "
        f"compute={bnb.bnb_4bit_compute_dtype}"
    )

    print(f"[bench] [{spec.model_id}] loading tokenizer: {spec.hf_name}")
    tokenizer = AutoTokenizer.from_pretrained(spec.hf_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    print(f"[bench] [{spec.model_id}] loading 4-bit base: {spec.hf_name}")
    model = AutoModelForCausalLM.from_pretrained(
        spec.hf_name,
        trust_remote_code=True,
        quantization_config=bnb,
        device_map={"": 0},
    )
    model.eval()
    model.config.use_cache = True

    if spec.adapter_path is not None:
        from peft import PeftModel

        adapter = Path(spec.adapter_path)
        if not adapter.exists():
            raise FileNotFoundError(f"LoRA adapter not found: {adapter}")
        print(f"[bench] [{spec.model_id}] attaching LoRA adapter: {adapter}")
        model = PeftModel.from_pretrained(model, str(adapter), is_trainable=False)
        model.eval()

    return model, tokenizer


def free_model(model) -> None:
    try:
        del model
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ── batched generation ──────────────────────────────────────────────────────


@torch.no_grad()
def _generate_subbatch(
    model,
    tokenizer,
    chunk: List[str],
    max_new_tokens: int,
) -> List[str]:
    """Run one forward generate on *chunk*; on CUDA OOM split and merge."""
    if not chunk:
        return []
    encoded = tokenizer(
        chunk,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(model.device)
    try:
        gen = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    except torch.cuda.OutOfMemoryError:  # type: ignore[attr-defined]
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if len(chunk) == 1:
            raise
        mid = (len(chunk) + 1) // 2
        print(
            f"[bench] CUDA OOM during generate (n={len(chunk)}); "
            f"splitting into {mid} + {len(chunk) - mid}"
        )
        return _generate_subbatch(model, tokenizer, chunk[:mid], max_new_tokens) + _generate_subbatch(
            model, tokenizer, chunk[mid:], max_new_tokens
        )
    prompt_lens = encoded["attention_mask"].sum(dim=1)
    texts: List[str] = []
    for i in range(gen.shape[0]):
        full_ids = gen[i].tolist()
        input_len = int(prompt_lens[i].item())
        new_ids = full_ids[input_len:] if input_len > 0 else full_ids
        texts.append(tokenizer.decode(new_ids, skip_special_tokens=True))
    return texts


@torch.no_grad()
def generate_batch(
    model,
    tokenizer,
    prompts: List[str],
    max_new_tokens: int,
    batch_size: int,
) -> List[str]:
    """Greedy decode prompts in chunks. Returns the *new* text per prompt."""
    out: List[str] = []
    for start in range(0, len(prompts), batch_size):
        chunk = prompts[start : start + batch_size]
        out.extend(_generate_subbatch(model, tokenizer, chunk, max_new_tokens))
    return out


# ── core evaluation loop ────────────────────────────────────────────────────


def _state_metrics(env: LongHorizonMemoryEnvironment) -> Dict[str, Any]:
    cap = env.MEMORY_CAPACITY
    n = len(env.memory)
    n_irrel = sum(1 for m in env.memory if not m.get("isRelevant", False))
    n_rel = n - n_irrel
    msg = env._current_message() or {}
    return {
        "memory_fill": n,
        "memory_capacity": cap,
        "memory_full": n >= cap,
        "memory_irrelevant_slots": n_irrel,
        "memory_relevant_slots": n_rel,
        "msg_relevant": bool(msg.get("isRelevant", False)),
    }


def evaluate_model_on_episodes(
    spec: ModelSpec,
    model,
    tokenizer,
    episode_ids: List[int],
    cfg: BenchConfig,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Evaluate a single model. Episodes are stepped forward in lockstep so we
    can batch generation across them. Returns (step_records, episode_summaries).
    """
    bs = cfg.batch_size_alt if spec.use_low_vram_batch else cfg.batch_size

    envs: List[LongHorizonMemoryEnvironment] = []
    for eid in episode_ids:
        env = LongHorizonMemoryEnvironment()
        env._episode_id_override = eid
        env._set_random_episode()
        envs.append(env)

    n_envs = len(envs)
    step_records: List[Dict[str, Any]] = []
    ep_step_count = [0] * n_envs
    ep_total_reward = [0.0] * n_envs
    ep_max_fill = [0] * n_envs
    ep_ops = [Counter() for _ in range(n_envs)]
    ep_errors = [Counter() for _ in range(n_envs)]
    ep_parse_status = [Counter() for _ in range(n_envs)]
    ep_correct_remove = [0] * n_envs
    ep_wrong_remove = [0] * n_envs

    round_idx = 0
    t0 = time.time()
    while any(not e._done for e in envs):
        active = [(i, e) for i, e in enumerate(envs) if not e._done]
        prompts = [build_chat_prompt(to_obs_dict(e), tokenizer) for _, e in active]
        completions = generate_batch(
            model,
            tokenizer,
            prompts,
            max_new_tokens=cfg.max_new_tokens,
            batch_size=bs,
        )

        for (i, env), completion_text in zip(active, completions):
            pre_state = _state_metrics(env)
            action, status = parse_completion(completion_text)
            applied = action if action is not None else fallback_action()

            obs = env.step(applied)
            reward = float(obs.reward)
            error = env.last_action_error  # set inside step before reward calc

            op = applied.operation
            ep_ops[i][op] += 1
            ep_parse_status[i][status] += 1
            if error:
                ep_errors[i][error] += 1
            if op == "remove":
                if reward > 0:
                    ep_correct_remove[i] += 1
                else:
                    ep_wrong_remove[i] += 1
            ep_total_reward[i] += reward
            ep_step_count[i] += 1
            ep_max_fill[i] = max(ep_max_fill[i], pre_state["memory_fill"])

            step_records.append(
                {
                    "model": spec.model_id,
                    "episode_id": episode_ids[i],
                    "step": ep_step_count[i],
                    "round": round_idx,
                    "pre_state": pre_state,
                    "completion_text": completion_text,
                    "parse_status": status,
                    "operation_requested": (action.operation if action else None),
                    "operation_applied": op,
                    "remove_index": (action.remove_index if action else None),
                    "task_reward": reward,
                    "env_error": error,
                    "memory_after": [
                        {"isRelevant": bool(m.get("isRelevant", False))}
                        for m in env.memory
                    ],
                }
            )

        round_idx += 1
        active_remaining = sum(1 for e in envs if not e._done)
        if round_idx % 10 == 0:
            elapsed = time.time() - t0
            print(
                f"[bench] [{spec.model_id}] round={round_idx} "
                f"active={active_remaining}/{n_envs} elapsed={elapsed:.1f}s"
            )

    elapsed = time.time() - t0
    print(f"[bench] [{spec.model_id}] finished {n_envs} eps in {elapsed:.1f}s")

    episode_summaries: List[Dict[str, Any]] = []
    for i, eid in enumerate(episode_ids):
        env = envs[i]
        n = ep_step_count[i] or 1
        # Final precision / recall / F1 from env state (running metrics).
        m = env._memory_stats()
        kept = len(env.memory)
        total_seen = max(1, env.total_relevant_seen)
        precision = m["correct"] / kept if kept > 0 else 1.0
        recall = m["correct"] / total_seen
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        episode_summaries.append(
            {
                "model": spec.model_id,
                "episode_id": eid,
                "domain": env.current_domain,
                "difficulty": env.current_difficulty,
                "n_messages": len(env.messages),
                "total_relevant_in_episode": env.total_relevant_in_episode,
                "n_steps": ep_step_count[i],
                "max_memory_fill": ep_max_fill[i],
                "final_memory_fill": kept,
                "final_memory_correct": m["correct"],
                "final_memory_incorrect": m["incorrect"],
                "final_precision": precision,
                "final_recall": recall,
                "final_f1": f1,
                "mean_step_reward": ep_total_reward[i] / n,
                "total_reward": ep_total_reward[i],
                "ops": dict(ep_ops[i]),
                "errors": dict(ep_errors[i]),
                "parse_status": dict(ep_parse_status[i]),
                "correct_remove": ep_correct_remove[i],
                "wrong_remove": ep_wrong_remove[i],
            }
        )

    return step_records, episode_summaries


# ── aggregation + comparison ────────────────────────────────────────────────


def summarize_model(model_id: str, step_records: List[Dict[str, Any]], ep_summaries: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not step_records:
        return {"model": model_id, "n_steps": 0}

    n_steps = len(step_records)
    parse_counter: Counter = Counter()
    op_counter: Counter = Counter()
    err_counter: Counter = Counter()
    correct_remove = 0
    wrong_remove = 0
    rewards: List[float] = []
    full_mem_steps = 0
    long_horizon_steps = 0
    for s in step_records:
        parse_counter[s["parse_status"]] += 1
        op_counter[s["operation_applied"]] += 1
        if s["env_error"]:
            err_counter[s["env_error"]] += 1
        if s["operation_applied"] == "remove":
            if s["task_reward"] > 0:
                correct_remove += 1
            else:
                wrong_remove += 1
        rewards.append(s["task_reward"])
        fill = s["pre_state"]["memory_fill"]
        if s["pre_state"]["memory_full"]:
            full_mem_steps += 1
        if fill >= 8:
            long_horizon_steps += 1

    n_ep = len(ep_summaries)
    f1s = [e["final_f1"] for e in ep_summaries]
    precs = [e["final_precision"] for e in ep_summaries]
    recs = [e["final_recall"] for e in ep_summaries]
    max_fills = [e["max_memory_fill"] for e in ep_summaries]

    summary = {
        "model": model_id,
        "n_episodes": n_ep,
        "n_steps": n_steps,
        "parse_rate": parse_counter["ok"] / n_steps,
        "parse_status": dict(parse_counter),
        "action_dist": {k: op_counter[k] / n_steps for k in ("add", "remove", "noop")},
        "remove_correctness": (
            correct_remove / max(1, correct_remove + wrong_remove)
        ),
        "remove_correct_count": correct_remove,
        "remove_wrong_count": wrong_remove,
        "env_errors": dict(err_counter),
        "mean_step_reward": sum(rewards) / n_steps,
        "median_step_reward": _median(rewards),
        "mean_final_f1": _mean(f1s),
        "median_final_f1": _median(f1s),
        "mean_final_precision": _mean(precs),
        "mean_final_recall": _mean(recs),
        "mean_max_memory_fill": _mean(max_fills),
        "long_horizon_step_pct": long_horizon_steps / n_steps,
        "memory_full_step_pct": full_mem_steps / n_steps,
    }
    return summary


def _mean(xs: List[float]) -> float:
    return sum(xs) / max(1, len(xs))


def _median(xs: List[float]) -> float:
    xs = sorted(xs)
    if not xs:
        return 0.0
    mid = len(xs) // 2
    if len(xs) % 2 == 1:
        return xs[mid]
    return 0.5 * (xs[mid - 1] + xs[mid])


def head_to_head(summaries: List[Dict[str, Any]], all_episodes: Dict[str, List[Dict[str, Any]]]) -> Dict[str, Any]:
    """Return per-episode wins (highest F1) per model and top deltas."""
    if "grpo_1.5b" not in all_episodes:
        return {}
    episode_ids = sorted({e["episode_id"] for e in all_episodes["grpo_1.5b"]})
    by_model_ep = {
        m: {e["episode_id"]: e for e in all_episodes[m]}
        for m in all_episodes
    }
    wins = Counter()
    grpo_vs_sft: List[Tuple[int, float, float, float]] = []
    grpo_vs_base: List[Tuple[int, float, float, float]] = []
    grpo_vs_large: List[Tuple[int, float, float, float]] = []
    large_id: Optional[str] = None
    for mid in ("base_3b", "base_7b", "base_05b", "base_custom"):
        if mid in all_episodes:
            large_id = mid
            break

    for eid in episode_ids:
        scores = {m: by_model_ep[m].get(eid, {}).get("final_f1", float("nan")) for m in all_episodes}
        best_model = max(scores, key=lambda m: (scores[m] if not math.isnan(scores[m]) else -1))
        wins[best_model] += 1
        if "sft_1.5b" in scores:
            grpo_vs_sft.append((eid, scores["grpo_1.5b"], scores["sft_1.5b"], scores["grpo_1.5b"] - scores["sft_1.5b"]))
        if "base_1.5b" in scores:
            grpo_vs_base.append((eid, scores["grpo_1.5b"], scores["base_1.5b"], scores["grpo_1.5b"] - scores["base_1.5b"]))
        if large_id and large_id in scores:
            grpo_vs_large.append(
                (
                    eid,
                    scores["grpo_1.5b"],
                    scores[large_id],
                    scores["grpo_1.5b"] - scores[large_id],
                )
            )

    grpo_vs_sft.sort(key=lambda r: r[3], reverse=True)
    grpo_vs_base.sort(key=lambda r: r[3], reverse=True)
    grpo_vs_large.sort(key=lambda r: r[3], reverse=True)

    return {
        "wins": dict(wins),
        "n_episodes": len(episode_ids),
        "large_baseline_id": large_id,
        "top_grpo_gains_vs_sft": grpo_vs_sft[:5],
        "top_sft_gains_over_grpo": list(reversed(grpo_vs_sft))[:5],
        "top_grpo_gains_vs_base": grpo_vs_base[:5],
        "top_grpo_gains_vs_large": grpo_vs_large[:5],
        # Kept for older result readers (same list when a large baseline is present)
        "top_grpo_gains_vs_7b": grpo_vs_large[:5] if large_id == "base_7b" else [],
    }


def collect_failure_modes(step_records: List[Dict[str, Any]], top_k: int = 5) -> Dict[str, Any]:
    """Return small samples of the most common failure modes for the report."""
    parse_failures = [s for s in step_records if s["parse_status"] != "ok"]
    env_errors = [s for s in step_records if s["env_error"]]

    parse_examples: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for s in parse_failures:
        if len(parse_examples[s["parse_status"]]) < top_k:
            parse_examples[s["parse_status"]].append(
                {
                    "episode_id": s["episode_id"],
                    "step": s["step"],
                    "completion_text": s["completion_text"][:200],
                }
            )
    error_examples: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for s in env_errors:
        if len(error_examples[s["env_error"]]) < top_k:
            error_examples[s["env_error"]].append(
                {
                    "episode_id": s["episode_id"],
                    "step": s["step"],
                    "operation_applied": s["operation_applied"],
                    "remove_index": s["remove_index"],
                    "memory_fill": s["pre_state"]["memory_fill"],
                    "completion_text": s["completion_text"][:200],
                }
            )
    return {
        "parse_failure_examples": dict(parse_examples),
        "env_error_examples": dict(error_examples),
    }


# ── reporting ───────────────────────────────────────────────────────────────


def render_table(summaries: List[Dict[str, Any]]) -> str:
    rows = [
        ("model", lambda s: s["model"]),
        ("n_eps", lambda s: f"{s['n_episodes']}"),
        ("n_steps", lambda s: f"{s['n_steps']}"),
        ("parse_ok %", lambda s: f"{s['parse_rate']*100:5.1f}"),
        ("add %", lambda s: f"{s['action_dist']['add']*100:5.1f}"),
        ("remove %", lambda s: f"{s['action_dist']['remove']*100:5.1f}"),
        ("noop %", lambda s: f"{s['action_dist']['noop']*100:5.1f}"),
        ("rem-correct %", lambda s: f"{s['remove_correctness']*100:5.1f}"),
        ("mean step rwd", lambda s: f"{s['mean_step_reward']:+.3f}"),
        ("mean final F1", lambda s: f"{s['mean_final_f1']:.3f}"),
        ("mean precision", lambda s: f"{s['mean_final_precision']:.3f}"),
        ("mean recall", lambda s: f"{s['mean_final_recall']:.3f}"),
        ("mean max fill", lambda s: f"{s['mean_max_memory_fill']:.1f}"),
        ("long-horiz %", lambda s: f"{s['long_horizon_step_pct']*100:5.1f}"),
        ("mem-full %", lambda s: f"{s['memory_full_step_pct']*100:5.1f}"),
    ]
    name_w = max(len(r[0]) for r in rows) + 1
    col_w = max(15, max((len(r[1](s)) for s in summaries for r in rows), default=15))
    lines = []
    header = " " * name_w + "".join(f"{s['model']:>{col_w}}" for s in summaries)
    sep = "-" * len(header)
    lines.append(sep)
    lines.append(header)
    lines.append(sep)
    for label, fn in rows:
        line = f"{label:<{name_w}}" + "".join(f"{fn(s):>{col_w}}" for s in summaries)
        lines.append(line)
    lines.append(sep)
    return "\n".join(lines)


def render_markdown_report(
    summaries: List[Dict[str, Any]],
    h2h: Dict[str, Any],
    failures: Dict[str, Dict[str, Any]],
    cfg: BenchConfig,
) -> str:
    lines: List[str] = []
    lines.append(f"# Long-Horizon Memory: model comparison ({cfg.run_id})")
    lines.append("")
    lines.append(f"- Episodes: **{cfg.n_episodes}** from `{cfg.episodes_file}`")
    lines.append(f"- Memory capacity: **{cfg.capacity}**")
    lines.append(f"- Decoding: greedy, max_new_tokens={cfg.max_new_tokens}")
    if h2h.get("large_baseline_id"):
        lines.append(
            f"- Larger untrained baseline in this run: **{h2h['large_baseline_id']}**"
        )
    lines.append("")
    lines.append("## Summary table")
    lines.append("")
    lines.append("```")
    lines.append(render_table(summaries))
    lines.append("```")
    lines.append("")

    if h2h:
        lines.append("## Head-to-head F1")
        lines.append("")
        lines.append(f"Per-episode wins (highest final_f1): `{h2h.get('wins', {})}` over {h2h['n_episodes']} episodes.")
        lines.append("")
        if h2h.get("top_grpo_gains_vs_sft"):
            lines.append("### Top 5 episodes where GRPO beat SFT (by F1 delta)")
            lines.append("")
            lines.append("| episode | grpo F1 | sft F1 | delta |")
            lines.append("|---------|---------|--------|-------|")
            for eid, g, s, d in h2h["top_grpo_gains_vs_sft"]:
                lines.append(f"| {eid} | {g:.3f} | {s:.3f} | {d:+.3f} |")
            lines.append("")
        if h2h.get("top_sft_gains_over_grpo"):
            lines.append("### Top 5 episodes where SFT beat GRPO (regressions to investigate)")
            lines.append("")
            lines.append("| episode | grpo F1 | sft F1 | delta |")
            lines.append("|---------|---------|--------|-------|")
            for eid, g, s, d in h2h["top_sft_gains_over_grpo"]:
                lines.append(f"| {eid} | {g:.3f} | {s:.3f} | {d:+.3f} |")
            lines.append("")
        if h2h.get("top_grpo_gains_vs_large") and h2h.get("large_baseline_id"):
            lb = h2h["large_baseline_id"]
            lines.append(f"### Top 5 episodes where GRPO 1.5B beat {lb} (larger untrained baseline)")
            lines.append("")
            lines.append(f"| episode | grpo F1 | {lb} F1 | delta |")
            lines.append("|---------|---------|--------|-------|")
            for eid, g, s, d in h2h["top_grpo_gains_vs_large"]:
                lines.append(f"| {eid} | {g:.3f} | {s:.3f} | {d:+.3f} |")
            lines.append("")

    lines.append("## Failure modes per model")
    lines.append("")
    for model_id, fails in failures.items():
        lines.append(f"### {model_id}")
        lines.append("")
        if fails.get("parse_failure_examples"):
            lines.append("**Parse failures** (sample):")
            lines.append("")
            for status, exs in fails["parse_failure_examples"].items():
                lines.append(f"- `{status}` (showing {len(exs)}):")
                for ex in exs:
                    lines.append(f"  - ep={ex['episode_id']} step={ex['step']} text={ex['completion_text']!r}")
            lines.append("")
        if fails.get("env_error_examples"):
            lines.append("**Env errors** (after parse, action was illegal):")
            lines.append("")
            for err, exs in fails["env_error_examples"].items():
                lines.append(f"- `{err}` (showing {len(exs)}):")
                for ex in exs:
                    lines.append(
                        f"  - ep={ex['episode_id']} step={ex['step']} "
                        f"op={ex['operation_applied']} remove_index={ex['remove_index']} "
                        f"mem_fill={ex['memory_fill']}"
                    )
            lines.append("")
    return "\n".join(lines)


def write_jsonl(path: Path, records: Iterable[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


# ── orchestration ───────────────────────────────────────────────────────────


def main() -> None:
    out_dir = Path(CFG.output_root) / CFG.run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[bench] writing results to {out_dir}")
    alt = MODEL_SPECS[-1]
    print(
        f"[bench] larger baseline: {alt.model_id}  ({alt.hf_name})  "
        f"low_vram_batch={alt.use_low_vram_batch}  "
        f"BATCH_SIZE={CFG.batch_size}  BATCH_SIZE_ALT={CFG.batch_size_alt}"
    )

    env_dir = HERE / "server"
    backup = use_episodes(env_dir, CFG.episodes_file)
    try:
        episode_ids = select_eval_episode_ids(CFG.n_episodes, CFG.seed)
        print(f"[bench] selected {len(episode_ids)} episodes: {episode_ids[:10]}{'...' if len(episode_ids) > 10 else ''}")
        write_json(out_dir / "config.json", {**asdict(CFG), "episode_ids": episode_ids})

        all_summaries: List[Dict[str, Any]] = []
        all_episode_summaries_by_model: Dict[str, List[Dict[str, Any]]] = {}
        all_failures: Dict[str, Dict[str, Any]] = {}

        for spec in selected_models():
            print(f"\n[bench] === {spec.model_id} ===  ({spec.description})")
            try:
                model, tokenizer = load_model_and_tokenizer(spec)
            except Exception as e:
                print(f"[bench] [{spec.model_id}] failed to load: {e!r}; skipping")
                continue
            try:
                steps, eps = evaluate_model_on_episodes(spec, model, tokenizer, episode_ids, CFG)
            finally:
                free_model(model)

            summary = summarize_model(spec.model_id, steps, eps)
            failures = collect_failure_modes(steps)

            write_jsonl(out_dir / f"{spec.model_id}_steps.jsonl", steps)
            write_jsonl(out_dir / f"{spec.model_id}_episodes.jsonl", eps)
            write_json(out_dir / f"{spec.model_id}_summary.json", summary)
            write_json(out_dir / f"{spec.model_id}_failures.json", failures)

            all_summaries.append(summary)
            all_episode_summaries_by_model[spec.model_id] = eps
            all_failures[spec.model_id] = failures

            print("\n[bench] partial summary so far:")
            print(render_table(all_summaries))

        h2h = head_to_head(all_summaries, all_episode_summaries_by_model)
        report_md = render_markdown_report(all_summaries, h2h, all_failures, CFG)
        table = render_table(all_summaries)

        write_json(out_dir / "comparison.json", {
            "config": asdict(CFG),
            "summaries": all_summaries,
            "head_to_head": h2h,
        })
        (out_dir / "comparison.md").write_text(report_md, encoding="utf-8")
        (out_dir / "comparison_table.txt").write_text(table, encoding="utf-8")

        print("\n" + "=" * 80)
        print("FINAL COMPARISON")
        print("=" * 80)
        print(table)
        print(f"\n[bench] Wrote: {out_dir / 'comparison.md'}")
    finally:
        restore_episodes(backup, env_dir)


if __name__ == "__main__":
    main()


[bench] writing results to /kaggle/working/open_env_meta_torch_comp/benchmark_results/run_20260426_050939
[bench] larger baseline: base_3b  (Qwen/Qwen2.5-3B-Instruct)  low_vram_batch=True  BATCH_SIZE=4  BATCH_SIZE_ALT=1
[bench] selected 20 episodes: [146, 355, 326, 247, 193, 116, 357, 319, 159, 101]...

[bench] === base_1.5b ===  (Qwen2.5-1.5B-Instruct, no fine-tuning. Floor for the action format.)
[bench] 4bit: type=nf4 double_quant=True compute=torch.bfloat16
[bench] [base_1.5b] loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


[bench] [base_1.5b] loading 4-bit base: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[bench] [base_1.5b] round=10 active=20/20 elapsed=83.0s
[bench] [base_1.5b] round=20 active=20/20 elapsed=222.8s
[bench] [base_1.5b] round=30 active=20/20 elapsed=375.8s
[bench] [base_1.5b] round=40 active=20/20 elapsed=528.2s
[bench] [base_1.5b] round=50 active=20/20 elapsed=683.3s
[bench] [base_1.5b] round=60 active=19/20 elapsed=834.2s
[bench] [base_1.5b] round=70 active=16/20 elapsed=976.9s
[bench] [base_1.5b] round=80 active=13/20 elapsed=1086.8s
[bench] [base_1.5b] round=90 active=12/20 elapsed=1180.9s
[bench] [base_1.5b] round=100 active=5/20 elapsed=1244.6s
[bench] [base_1.5b] round=110 active=0/20 elapsed=1270.0s
[bench] [base_1.5b] finished 20 eps in 1270.0s

[bench] partial summary so far:
------------------------------
                     base_1.5b
------------------------------
model                base_1.5b
n_eps                       20
n_steps                   1768
parse_ok %               100.0
add %                     94.8
remove %                   5.2
noop %     

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[bench] [sft_1.5b] attaching LoRA adapter: /kaggle/working/open_env_meta_torch_comp/memory_action_sft_qwen15b


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


[bench] [sft_1.5b] round=10 active=20/20 elapsed=99.3s
[bench] [sft_1.5b] round=20 active=20/20 elapsed=245.1s
[bench] [sft_1.5b] round=30 active=20/20 elapsed=411.2s
[bench] [sft_1.5b] round=40 active=20/20 elapsed=577.7s
[bench] [sft_1.5b] round=50 active=20/20 elapsed=743.6s
